[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsonmen/bias-and-variance/blob/main/computer-vision/ImageClassification/CatsvsDogs/ResNetTransferLearning.ipynb)

# Abstract

- Goal: Fine-Tune ResNet18 on Cats vs Dogs dataset and reach good accuracy score

- Dataset: [Cats and Dogs Classification Dataset (Kaggle)](https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset)

- Project Details:

    I'm used pre-trained ResNet18 on IMAGENET1K dataset and fine-tune it for binary classification task on Cats vs Dogs dataset. Also i make form for prediction your own image or select image from dataset.

  
- Best result: 0.9642 (Accuracy Score of ResNet18)

- Sections:
    - [Imports](#Imports)
    - [Model Parameters](#Model-Parameters)
    - [Utils](#Utils)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
    - [Prediction](#Prediction)
        - [Load Model](#Load-Model)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

## Problems & Solutions
There was no problems. It's easy just load pre-trained model, prepare dataset, train model and you get a good result.

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install torch torchvision matplotlib pillow numpy tqdm ipywidgets ipython

In [23]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/bhavikjikadara/dog-and-cat-classification-dataset
!unzip -qq ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/PetImages ./data/catsvsdogs

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  775M  100  775M    0     0  10.6M      0  0:01:12  0:01:12 --:--:-- 10.8M


In [17]:
# Model Downloading
!mkdir models
from huggingface_hub import snapshot_download

PROJECT_NAME = "ImageClassificaitionCatsVSDogs"
MODEL_FOLDER = "resnet18"

repo_id = f"jsonmen/{PROJECT_NAME}"

snapshot_download(
    repo_id=repo_id,
    local_dir="./models",
    allow_patterns=[f"{MODEL_FOLDER}/*"],
    token=False  # No token needed for public repos
)
print(f"Downloaded models folder from {repo_id} to ./models")

mkdir: cannot create directory ‘models’: File exists


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

resnet18_model.pt:   0%|          | 0.00/44.8M [00:00<?, ?B/s]

Downloaded models folder from jsonmen/ImageClassificaitionCatsVSDogs to ./models


# Imports

In [5]:
import torch
from torch import nn, Tensor
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import v2
from PIL import Image
import numpy as np
from tqdm import tqdm
import sys
import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# Model Parameters

In [6]:
BATCH_SIZE = 32
LR = 3e-4
EPOCH = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

# Utils

In [7]:
def binary_accuracy(y_pred, y_true):
    """
    Compute accuracy for binary classification.
    """
    # sigmoid → probabilities
    probs = torch.sigmoid(y_pred)
    preds = torch.round(probs)
    correct = (preds == y_true).float()
    acc = correct.sum() / len(correct)
    return acc

def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    """
    Train model for one epoch.
    """
    model.train()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for inputs, targets in progress_bar:
        inputs = inputs.to(device)
        targets = targets.to(device).float().unsqueeze(1)  # Shape: (batch, 1)

        optimizer.zero_grad()
        outputs = model(inputs)

        loss = loss_fn(outputs, targets)
        acc = binary_accuracy(outputs, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_acc += acc.item() * inputs.size(0)

        avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
        avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
        progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, loss_fn, device):
    """
    Evaluate model.
    """
    model.eval()
    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(dataloader, desc="Val", leave=False)

    with torch.no_grad():
        for inputs, targets in progress_bar:
            inputs = inputs.to(device)
            targets = targets.to(device).float().unsqueeze(1)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            acc = binary_accuracy(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            running_acc += acc.item() * inputs.size(0)

            avg_loss = running_loss / ((progress_bar.n + 1) * inputs.size(0))
            avg_acc = running_acc / ((progress_bar.n + 1) * inputs.size(0))
            progress_bar.set_postfix(loss=avg_loss, acc=avg_acc)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_acc / len(dataloader.dataset)
    return epoch_loss, epoch_acc


def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs):
    """
    Run full training loop.
    """
    # Example usage:
    # train_loop(
    #     model,
    #     train_loader,
    #     val_loader,
    #     optimizer,
    #     loss_fn,
    #     DEVICE,
    #     EPOCH
    # )
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, loss_fn, device
        )

        val_loss, val_acc = evaluate(
            model, val_loader, loss_fn, device
        )

        print(
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

In [8]:
import warnings
warnings.filterwarnings(
    action='ignore',
    category=UserWarning,
)


# Dataset

In [9]:
class SafeImageFolder(torchvision.datasets.ImageFolder): # This class its ImageFolder but with broken image filtering 
    def __init__(self, root, transform=None, target_transform=None):
        super().__init__(root, transform=transform, target_transform=target_transform)
        self.samples = [
            (path, class_idx) for path, class_idx in self.samples
            if self._is_valid_image(path)
        ]

    def _is_valid_image(self, path):
        try:
            with Image.open(path) as img:
                img.verify()
            return True
        except Exception:
            return False

In [10]:
image_transforms = v2.Compose([v2.Resize((224, 224)), 
                        v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]) 

In [11]:
dataset = SafeImageFolder("./data/catsvsdogs", transform=image_transforms, target_transform=lambda x: torch.tensor(x, dtype=torch.float32))
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42))

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Modeling


Few words about **ResNet‑18**:
**ResNet‑18** is short for **Residual Network** with 18 layers.
It introduces **skip (identity) connections** that let the network learn residual functions instead of direct mappings, which greatly eases training of deep nets.

For a classification task, i simply replace final fully‑connected layer with similar fully‑connected layer (output dim only changed to 1)

**Image that makes sense of ResNet‑18’s residual block:**

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/ba/ResBlock.png/250px-ResBlock.png" alt="ResNet residual block diagram" width="500"/>

**Image comparing ResNet‑18 vs. a plain network:**

<img src="https://miro.medium.com/v2/resize:fit:1400/1*Cf7kM-zS7GbtQd6ljQkEsg.png" alt="ResNet vs plain CNN" width="500"/>

In [36]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
num_in_features = model.fc.in_features
model.fc = nn.Linear(in_features=num_in_features, out_features=1, bias=True)
for param in model.parameters():
    param.requires_grad = True
model = model.to(DEVICE)

In [38]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [39]:
train_loop(model, train_dataloader, val_dataloader, optimizer, loss_fn, DEVICE, EPOCH)


Epoch 1/2


Train Loss: 0.0871 | Train Acc: 0.9653 | Val Loss: 0.0650 | Val Acc: 0.9762

Epoch 2/2


Train Loss: 0.0543 | Train Acc: 0.9788 | Val Loss: 0.0888 | Val Acc: 0.9642


In [40]:
torch.save(model.state_dict(), "./models/resnet18/resnet18_model.pt")

# Prediction

## Load Model

In [12]:
trained_model = torchvision.models.resnet18(weights=None)
num_in_features = trained_model.fc.in_features
trained_model.fc = nn.Linear(in_features=num_in_features, out_features=1, bias=True)
trained_model.load_state_dict(torch.load("./models/resnet18/resnet18_model.pt", map_location=torch.device('cpu')))
trained_model = trained_model.cpu()
trained_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Setup Form

In [13]:
t1_output = widgets.Output()

image_id = widgets.BoundedIntText(
    value=13,
    min=0,
    max=len(val_dataset),
    step=1,
    description='Image id from dataset:',
    disabled=False,
    style={'description_width': 'initial'}
)
def t1_func(b):
    image_id_i = image_id.value
    cat_or_dog = lambda x: "Cat" if torch.tanh(x).item() <= 0 else "Dog"  
    with t1_output:
        clear_output()
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(val_dataset[image_id_i][0].permute(1, 2, 0).numpy())
        plt.title("Selected image")
        plt.xticks([])
        plt.yticks([])
        plt.subplot(1, 2, 2)
        plt.axis('off')
        prediction = trained_model(val_dataset[image_id_i][0].unsqueeze(0))
        info = f"""Raw Model Prediction: {prediction.item():.3f}

Model Prediction in Words: {cat_or_dog(prediction)}
Model Prediction Confidence: {torch.tanh(prediction).abs().item():.2%}"""

        plt.text(0, 1, info, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
        

t1_button = widgets.Button(description="Classify")
t1_button.on_click(t1_func)
t1 = widgets.VBox([image_id, t1_button, t1_output])

In [14]:
t2_output = widgets.Output()

image_file = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    desctiption="Image file Upload: "
)
def t2_func(b):
    filename = list(image_file.value.keys())[0]
    bytes_content = image_file.value[filename]['content']
    pil_image = Image.open(io.BytesIO(bytes_content))
    cat_or_dog = lambda x: "Cat" if torch.tanh(x).item() <= 0 else "Dog" 
    with t2_output:
        clear_output()
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.imshow(pil_image)
        plt.title("Selected image")
        plt.xticks([])
        plt.yticks([])
        plt.subplot(1, 2, 2)
        plt.axis('off')
        prediction = trained_model(image_transforms(pil_image).unsqueeze(0))
        info = f"""Raw Model Prediction: {prediction.item():.3f}

Model Prediction in Words: {cat_or_dog(prediction)}
Model Prediction Confidence: {torch.tanh(prediction).abs().item():.2%}"""

        plt.text(0, 1, info, fontsize=12, va='top')
        plt.tight_layout()
        plt.show()
t2_button = widgets.Button(description="Classify")
t2_button.on_click(t2_func)

t2 = widgets.VBox([image_file, t2_button, t2_output])

In [15]:
form = widgets.Tab()
form.children = [t1, t2]
form.set_title(0, 'Select from dataset')
form.set_title(1, 'Upload image file')

## Prediction Form

In [16]:
display(form)